In [13]:
# =========================
# 1. Imports & Environment Setup
# =========================

from dotenv import load_dotenv
import os
import json
import requests
from datetime import datetime, date
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd
from litellm import completion

load_dotenv()  # e.g. OPENAI_API_KEY / LITELLM_API_KEY, etc.

# Base endpoint: this week's UMich events
EVENTS_URL = "https://events.umich.edu/week/json?v=2"

# LLM model to use
DEFAULT_MODEL = "gpt-4o-mini"

In [14]:
# =========================
# 2. Fetch Events from UMich
# =========================

def fetch_umich_events():
    """
    Fetch events from the UMich events API.

    Currently uses the 'week' endpoint defined in EVENTS_URL.
    Returns a list of event dicts.
    """
    resp = requests.get(EVENTS_URL, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    # API sometimes returns a list, sometimes wraps in {"events": [...]}
    if isinstance(data, list):
        return data
    if isinstance(data, dict) and "events" in data:
        return data["events"]

    print("Unexpected events JSON structure:", type(data))
    return []


In [15]:
# =========================
# 3. Filter Events (today and future only)
# =========================

def filter_future_events(events, from_date=None):
    """
    Filter events to only those that occur on or after `from_date`.

    - from_date: a datetime.date object. If None, defaults to today.
    - Uses event['date_start'] in 'YYYY-MM-DD' format.
    """
    if from_date is None:
        from_date = date.today()

    def is_future_event(ev):
        d = ev.get("date_start")
        if not d:
            return False
        try:
            event_date = datetime.strptime(d, "%Y-%m-%d").date()
            return event_date >= from_date
        except Exception:
            return False

    filtered = [ev for ev in events if is_future_event(ev)]
    return filtered


In [16]:
# =========================
# 4. LLM Analysis (classify + extract free stuff)
# =========================

def run_completion(model, messages, **kwargs):
    """
    Thin wrapper around litellm.completion that returns the assistant content.
    """
    try:
        response = completion(
            model=model,
            messages=messages,
            **kwargs
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error fetching response from LLM: {e}")
        return None


def analyze_event_with_llm(event, model=DEFAULT_MODEL, **kwargs):
    """
    For a single event, use the LLM to:
      - classify: free_food / free_snacks / other_free / none
      - extract: what exactly is free (short + longer explanation)

    Returns a dict:
    {
      "label": "free_food" | "free_snacks" | "other_free" | "none",
      "free_item": str,
      "free_details": str,
    }
    """
    title = (
        event.get("combined_title")
        or event.get("event_title")
        or ""
    )
    desc = event.get("description", "") or ""
    location = (
        event.get("location_name")
        or event.get("building_name")
        or ""
    )
    cost = event.get("cost", "") or ""
    tags = " ".join(event.get("tags", []))

    event_text = (
        f"Title: {title}\n"
        f"Location: {location}\n"
        f"Cost: {cost}\n"
        f"Tags: {tags}\n"
        f"Description: {desc}"
    )

    system_prompt = (
        "You are an assistant analyzing a university event listing. "
        "You must detect if there is anything explicitly free, and if so, what.\n\n"
        "Classification rules:\n"
        "- 'free_food'   -> free meals (pizza, dinner, lunch, breakfast, etc.).\n"
        "- 'free_snacks' -> free snacks/drinks (cookies, donuts, chips, candy, coffee, tea, hot cocoa, cider, etc.).\n"
        "- 'other_free'  -> other free things (free entry/ticket, free merch, free workshop, free exhibit, etc.).\n"
        "- 'none'        -> no clearly free item mentioned.\n\n"
        "Be conservative: if it's not clearly free, choose 'none'.\n\n"
        "Return ONLY a JSON object with this shape:\n"
        "{\n"
        "  \"label\": \"free_food\" | \"free_snacks\" | \"other_free\" | \"none\",\n"
        "  \"free_item\": \"a short phrase naming what is free (or empty if none)\",\n"
        "  \"free_details\": \"1–2 sentence explanation of the free offering (or empty if none)\"\n"
        "}"
    )

    user_prompt = (
        f"Here are the event details:\n\n{event_text}\n\n"
        "Classify and extract according to the instructions."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",  "content": user_prompt},
    ]

    content = run_completion(model, messages, **kwargs)
    if not content:
        return {"label": "none", "free_item": "", "free_details": ""}

    # Try to parse JSON response
    try:
        data = json.loads(content)
        label = data.get("label", "none")
        free_item = data.get("free_item", "") or ""
        free_details = data.get("free_details", "") or ""

        if label not in {"free_food", "free_snacks", "other_free", "none"}:
            label = "none"

        return {
            "label": label,
            "free_item": free_item.strip(),
            "free_details": free_details.strip(),
        }
    except Exception:
        # Fallback if the model messes up JSON
        text = content.strip().lower()
        if "free_food" in text:
            label = "free_food"
        elif "free_snacks" in text:
            label = "free_snacks"
        elif "other_free" in text:
            label = "other_free"
        elif "none" in text:
            label = "none"
        else:
            label = "none"

        return {
            "label": label,
            "free_item": "",
            "free_details": content.strip(),
        }


In [17]:
# =========================
# 5. Concurrent LLM Processing
# =========================

def process_events_concurrent(events, model=DEFAULT_MODEL, max_workers=20, **kwargs):
    """
    Analyze all events concurrently with LLM and build enriched rows
    for events that have something free.

    Returns:
      - enriched_rows: list of dicts (one per free-related event)
      - buckets: dict[label] -> list of enriched rows
    """
    enriched_rows = []
    futures = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for idx, ev in enumerate(events):
            fut = executor.submit(analyze_event_with_llm, ev, model, **kwargs)
            futures[fut] = idx

        for fut in as_completed(futures):
            idx = futures[fut]
            ev = events[idx]

            try:
                analysis = fut.result()
            except Exception as e:
                print(f"Error analyzing event at index {idx}: {e}")
                analysis = {"label": "none", "free_item": "", "free_details": ""}

            label = analysis.get("label", "none")
            if label == "none":
                continue  # skip events with nothing free

            # Build enriched row with metadata
            sponsors_list = ev.get("sponsors", []) or []
            sponsors = "; ".join(
                s.get("group_name", "") for s in sponsors_list if s.get("group_name")
            )

            row = {
                "id": ev.get("id", ""),
                "label": label,
                "title": ev.get("combined_title") or ev.get("event_title") or "",
                "free_item": analysis.get("free_item", ""),
                "free_details": analysis.get("free_details", ""),
                "date_start": ev.get("date_start", ""),
                "time_start": ev.get("time_start", ""),
                "date_end": ev.get("date_end", ""),
                "time_end": ev.get("time_end", ""),
                "time_zone": ev.get("time_zone", ""),
                "location_name": ev.get("location_name", ""),
                "building_name": ev.get("building_name", ""),
                "room": ev.get("room", ""),
                "cost": ev.get("cost", ""),
                "tags": ", ".join(ev.get("tags", [])),
                "organizers": sponsors,
                "permalink": ev.get("permalink", ""),
                "website": ev.get("website", ""),
            }

            enriched_rows.append(row)

    # Build buckets for nicer printing
    buckets = {
        "free_food":   [],
        "free_snacks": [],
        "other_free":  [],
    }
    for row in enriched_rows:
        buckets[row["label"]].append(row)

    return enriched_rows, buckets


In [18]:
# =========================
# 6. Persistent State: Detect NEW Free Events
# =========================

from pathlib import Path

STATE_FILE = Path("free_events_state.json")

def detect_new_events(enriched_rows, state_file: Path = STATE_FILE):
    """
    Compare current enriched_rows against previously seen free-event IDs.

    Returns:
      - new_events: list of rows that are newly free this run
      - removed_event_ids: set of IDs that disappeared
      - summary: dict with counts
    Also updates the state file with the current IDs.
    """
    # Load previous known free-event IDs
    if state_file.exists():
        with open(state_file, "r") as f:
            prev_state = json.load(f)
        prev_ids = set(prev_state.get("event_ids", []))
    else:
        prev_state = {}
        prev_ids = set()

    # Current free-event IDs
    current_ids = set(e["id"] for e in enriched_rows)

    # Compute diffs
    new_event_ids = current_ids - prev_ids
    removed_event_ids = prev_ids - current_ids
    unchanged_event_ids = current_ids & prev_ids

    # Build list of only new events
    new_events = [e for e in enriched_rows if e["id"] in new_event_ids]

    # Update state
    with open(state_file, "w") as f:
        json.dump({"event_ids": list(current_ids)}, f, indent=2)

    summary = {
        "previous_count": len(prev_ids),
        "current_count": len(current_ids),
        "new_count": len(new_event_ids),
        "removed_count": len(removed_event_ids),
        "unchanged_count": len(unchanged_event_ids),
    }

    return new_events, removed_event_ids, summary


In [19]:
# =========================
# 7. Pretty Printing
# =========================

def pretty_print_bucket(name, rows):
    print(f"=== {name} ===")
    if not rows:
        print("  (none)\n")
        return

    for r in rows:
        when = f"{r['date_start']} {r['time_start']}".strip()
        where_parts = [r["location_name"], r["building_name"], r["room"]]
        where = ", ".join([p for p in where_parts if p])
        org = r["organizers"] or "Unknown organizer"
        free_info = r["free_item"] or "[unspecified free item]"

        print(f"- {r['title']}")
        print(f"  • Free: {free_info}")
        print(f"  • When: {when} ({r['time_zone']})")
        print(f"  • Where: {where}")
        print(f"  • Organizers: {org}")
        print(f"  • Details: {r['free_details']}")
        print(f"  • Link: {r['permalink']}\n")


def pretty_print_new_events(new_events):
    print("\n=== NEW FREE EVENTS (compared to last run) ===")
    if not new_events:
        print("No new free events — everything was seen before.\n")
        return

    for r in new_events:
        when = f"{r['date_start']} {r['time_start']}".strip()
        where_parts = [r["location_name"], r["building_name"], r["room"]]
        where = ", ".join([p for p in where_parts if p])
        org = r["organizers"] or "Unknown organizer"
        free_info = r["free_item"] or "[unspecified free item]"

        print(f"- {r['title']}")
        print(f"  • Free: {free_info}")
        print(f"  • When: {when} ({r['time_zone']})")
        print(f"  • Where: {where}")
        print(f"  • Organizers: {org}")
        print(f"  • Details: {r['free_details']}")
        print(f"  • Link: {r['permalink']}\n")


In [20]:
# =========================
# 8. Save to CSV (with date range in filename)
# =========================

def save_events_csv(rows, all_events, prefix="umich_free_events"):
    """
    Save given rows to a CSV file.
    Filename includes the min/max date_start across all_events.

    Returns:
      - csv_path: str
    """
    if not rows:
        print("No rows to save, skipping CSV.")
        return None

    df = pd.DataFrame(rows)

    # derive date range
    all_dates = [e.get("date_start") for e in all_events if e.get("date_start")]
    if all_dates:
        week_start = min(all_dates)
        week_end = max(all_dates)
        csv_path = f"{prefix}_{week_start}_to_{week_end}.csv"
    else:
        csv_path = f"{prefix}_nodates.csv"

    df.to_csv(csv_path, index=False)
    print(f"Saved {len(rows)} rows to CSV: {csv_path}")
    return csv_path


In [ ]:
# =========================
# 9. Orchestration / Main Flow
# =========================

# 1) Fetch raw events
events_raw = fetch_umich_events()
print(f"Fetched {len(events_raw)} events total")

# 2) Filter to today and future
events = filter_future_events(events_raw)
print(f"Filtered to {len(events)} events occurring today or in the future\n")

# 3) Run concurrent LLM analysis
enriched_rows, buckets = process_events_concurrent(
    events,
    model=DEFAULT_MODEL,
    max_workers=50,   # you can bump this carefully
    temperature=0     # deterministic
)

print(f"Found {len(enriched_rows)} events with something free.\n")

# 4) Compare with previous state to find NEW free events
new_events, removed_event_ids, summary = detect_new_events(enriched_rows)

print("=== STATE SUMMARY ===")
print(f"Previously known free events: {summary['previous_count']}")
print(f"Current free events:         {summary['current_count']}")
print(f"New free events:             {summary['new_count']}")
print(f"Removed free events:         {summary['removed_count']}\n")

# 5) Pretty-print by bucket (all current free events)
pretty_print_bucket("Events with FREE FOOD", buckets["free_food"])
pretty_print_bucket("Events with FREE SNACKS", buckets["free_snacks"])
pretty_print_bucket("Events with OTHER FREE STUFF", buckets["other_free"])

# 6) Pretty-print ONLY new free events since last run
pretty_print_new_events(new_events)

# 7) Save ONLY new free events to CSV
csv_path = save_events_csv(new_events, events, prefix="umich_NEW_free_events")


Fetched 683 events total
Filtered to 357 events occurring today or in the future

Found 125 events with something free.

=== STATE SUMMARY ===
Previously known free events: 0
Current free events:         125
New free events:             125
Removed free events:         0

=== Events with FREE FOOD ===
- Student Caregiver Appreciation Week
  • Free: breakfast
  • When: 2025-11-13 00:00:00 (America/Detroit)
  • Where: 
  • Organizers: CEW+; University Career Center UCC; Student Life; LSA Opportunity Hub; LSA Transfer Student Center
  • Details: Join us for a breakfast and caffeine at the Student Caregiver Breakfast on November 10, 2025, as part of the kickoff for Student Caregiver Appreciation Week.
  • Link: http://events.umich.edu/event/140630

- Culture Under the Microscope Series: Thriving Together: Navigating Career and Community as an International Trainee
  • Free: lunch
  • When: 2025-11-13 11:00:00 (America/Detroit)
  • Where: THSL 2994
  • Organizers: Sessions @ Michigan
  • De

In [22]:
# =========================
# Professional Relevance Classifier (LLM-based)
# =========================

def analyze_professional_relevance(event, model=DEFAULT_MODEL, **kwargs):
    """
    Classify whether an event is professionally helpful to you
    (career, product management, tech, entrepreneurship, design, research,
    networking, academic development, internships, jobs, leadership, etc.).

    Returns:
    {
      "is_helpful": bool,
      "reason": str,   # short explanation from LLM
      "category": str  # e.g., 'career', 'networking', 'research', etc.
    }
    """

    title = (
        event.get("combined_title")
        or event.get("event_title")
        or ""
    )
    desc = event.get("description", "") or ""
    tags = " ".join(event.get("tags", []))
    orgs = "; ".join(
        o.get("group_name", "") 
        for o in (event.get("sponsors") or [])
        if o.get("group_name")
    )

    event_text = (
        f"Title: {title}\n"
        f"Description: {desc}\n"
        f"Tags: {tags}\n"
        f"Organizers: {orgs}\n"
    )

    system_prompt = (
        "You are a professional advisor evaluating whether a university event "
        "is professionally helpful to a young tech/Product Management-oriented "
        "graduate student who is interested in:\n"
        "- product management\n"
        "- UX design\n"
        "- AI and software engineering\n"
        "- entrepreneurship & startups\n"
        "- research and academic development\n"
        "- networking and leadership\n"
        "- career growth and professional skills\n\n"
        "Evaluate the event and classify it as either:\n"
        "  helpful: true/false\n"
        "  category: career | networking | PM/tech | research | academic | leadership | entrepreneurship | general\n"
        "  reason:  a short explanation why.\n\n"
        "Return ONLY JSON in this structure:\n"
        "{\n"
        "  \"is_helpful\": true/false,\n"
        "  \"category\": \"career\" | \"networking\" | \"PM/tech\" | \"research\" | \"academic\" | \"leadership\" | \"entrepreneurship\" | \"general\",\n"
        "  \"reason\": \"...\"\n"
        "}"
    )

    user_prompt = (
        f"Evaluate this event:\n\n{event_text}\n"
        "Is it professionally helpful to this person?"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",  "content": user_prompt},
    ]

    content = run_completion(model, messages, **kwargs)
    if not content:
        return {"is_helpful": False, "category": "general", "reason": "No response"}

    # Try parse JSON
    try:
        parsed = json.loads(content)
        return {
            "is_helpful": parsed.get("is_helpful", False),
            "category": parsed.get("category", "general"),
            "reason": parsed.get("reason", "").strip(),
        }
    except:
        # Fallback
        text = content.lower()
        return {
            "is_helpful": "true" in text,
            "category": "general",
            "reason": content,
        }


def process_professional_events_concurrent(events, model=DEFAULT_MODEL, max_workers=20, **kwargs):
    """
    Concurrently classify professional helpful events.
    Returns:
      helpful_events: list of enriched rows
      categories: dict with events sorted by category
    """
    helpful_events = []
    futures = {}

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for idx, ev in enumerate(events):
            fut = executor.submit(analyze_professional_relevance, ev, model, **kwargs)
            futures[fut] = idx

        for fut in as_completed(futures):
            idx = futures[fut]
            ev = events[idx]

            try:
                analysis = fut.result()
            except Exception as e:
                print(f"Error evaluating relevance for event {idx}: {e}")
                analysis = {"is_helpful": False, "category": "general", "reason": ""}

            if not analysis["is_helpful"]:
                continue

            # Build enriched row
            row = {
                "id": ev.get("id", ""),
                "title": ev.get("combined_title") or ev.get("event_title") or "",
                "date_start": ev.get("date_start", ""),
                "time_start": ev.get("time_start", ""),
                "location": ev.get("location_name", "") or ev.get("building_name", ""),
                "organizers": "; ".join(
                    o.get("group_name", "")
                    for o in (ev.get("sponsors") or [])
                    if o.get("group_name")
                ),
                "category": analysis["category"],
                "reason": analysis["reason"],
                "link": ev.get("permalink", ""),
            }

            helpful_events.append(row)

    # Group by category
    categories = {}
    for r in helpful_events:
        categories.setdefault(r["category"], []).append(r)

    return helpful_events, categories


In [ ]:
# =========================
# Professional Events Runner
# =========================

# 1) Fetch raw events
events_raw = fetch_umich_events()
print(f"Fetched {len(events_raw)} events")

# 2) Filter to today + future
events = filter_future_events(events_raw)
print(f"{len(events)} events are today or later\n")

# 3) Analyze professional helpfulness (LLM, concurrent)
helpful_events, categories = process_professional_events_concurrent(
    events,
    model=DEFAULT_MODEL,
    max_workers=50,
    temperature=0
)

print(f"\n=== Professionally Helpful Events: {len(helpful_events)} ===\n")

# 4) Pretty print by category
for cat, rows in categories.items():
    print(f"--- {cat.upper()} ({len(rows)}) ---")
    for r in rows:
        print(f"- {r['title']}")
        print(f"  • Date: {r['date_start']} {r['time_start']}")
        print(f"  • Where: {r['location']}")
        print(f"  • Organizers: {r['organizers']}")
        print(f"  • Why helpful: {r['reason']}")
        print(f"  • Link: {r['link']}\n")

# 5) Save CSV
df_prof = pd.DataFrame(helpful_events)
csv_path = "umich_professionally_helpful_events.csv"
df_prof.to_csv(csv_path, index=False)
print(f"\nSaved professionally helpful events CSV to: {csv_path}")


Fetched 683 events
357 events are today or later


=== Professionally Helpful Events: 39 ===

--- NETWORKING (10) ---
- Join SOCHI Email List!
  • Date: 2025-11-13 00:00:00
  • Where: SOCHI
  • Organizers: Maize Pages Student Organizations
  • Why helpful: Joining the email list provides access to events and networking opportunities related to HCI and UX/UI, which are relevant to the student's interests in product management and UX design.
  • Link: http://events.umich.edu/event/132112

- SLB Faculty-Student Lunch with Assistant Prof. Sushil Varma
  • Date: 2025-11-13 11:00:00
  • Where: Industrial and Operations Engineering Building
  • Organizers: Industrial & Operations Engineering
  • Why helpful: The event provides an opportunity for students to network with a faculty member, which can lead to mentorship, guidance, and potential collaboration in areas related to product management, UX design, and entrepreneurship.
  • Link: http://events.umich.edu/event/141573

- Info Session: Sil

In [25]:
import streamlit as st
import pandas as pd
from datetime import date

# ⬇️ IMPORTANT: make sure these are imported from your module
# or defined above this block in the same file.
#
# from your_module import (
#     fetch_umich_events,
#     filter_future_events,
#     process_events_concurrent,
#     process_professional_events_concurrent,
#     DEFAULT_MODEL,
# )

st.set_page_config(
    page_title="UMich Events Radar",
    page_icon="🔍",
    layout="wide",
)

st.title("UMich Events Radar 🔍")
st.caption("Scans UMich events for free stuff & professionally helpful opportunities.")

# Sidebar controls
st.sidebar.header("Settings")

max_workers = st.sidebar.slider("Max concurrent LLM workers", 5, 80, 20, step=5)
temperature = st.sidebar.slider("Model temperature", 0.0, 1.0, 0.0, step=0.1)

show_free = st.sidebar.checkbox("Show free / free-ish events", value=True)
show_prof = st.sidebar.checkbox("Show professionally helpful events", value=True)

run_button = st.sidebar.button("Run scan")

st.sidebar.markdown("---")
st.sidebar.write("Today:", date.today().isoformat())


if run_button:
    with st.spinner("Fetching and filtering events from UMich API..."):
        events_raw = fetch_umich_events()
        events = filter_future_events(events_raw)

    st.write(f"Fetched **{len(events_raw)}** events total.")
    st.write(f"Filtered to **{len(events)}** events happening today or later.\n")

    tabs = []
    if show_free:
        tabs.append("🍕 Free / Free-ish")
    if show_prof:
        tabs.append("💼 Professionally Helpful")

    if not tabs:
        st.warning("Enable at least one option (free events or professional events) in the sidebar.")
    else:
        tab_objs = st.tabs(tabs)

        # ---------- FREE EVENTS TAB ----------
        if show_free:
            tab_idx = tabs.index("🍕 Free / Free-ish")
            with tab_objs[tab_idx]:
                st.subheader("🍕 Events with Free Food / Snacks / Other Free Stuff")
                with st.spinner("Scanning events for free items..."):
                    enriched_rows, buckets = process_events_concurrent(
                        events,
                        model=DEFAULT_MODEL,
                        max_workers=max_workers,
                        temperature=temperature,
                    )

                st.write(f"Found **{len(enriched_rows)}** events with something free.")

                # Summary metrics
                col1, col2, col3 = st.columns(3)
                col1.metric("Free food", len(buckets["free_food"]))
                col2.metric("Free snacks", len(buckets["free_snacks"]))
                col3.metric("Other free stuff", len(buckets["other_free"]))

                # Full table
                df_free = pd.DataFrame(enriched_rows)
                with st.expander("View all free-related events as a table", expanded=True):
                    st.dataframe(df_free, use_container_width=True)

                # Nice grouped view
                st.markdown("#### Grouped by Type of Free Stuff")
                for label, label_title in [
                    ("free_food", "🍽️ Free Food"),
                    ("free_snacks", "☕ Free Snacks"),
                    ("other_free", "🎁 Other Free Stuff"),
                ]:
                    rows = buckets[label]
                    if not rows:
                        continue
                    st.markdown(f"**{label_title} ({len(rows)})**")
                    for r in rows:
                        with st.expander(r["title"]):
                            when = f"{r['date_start']} {r['time_start']}".strip()
                            where_parts = [r["location_name"], r["building_name"], r["room"]]
                            where = ", ".join([p for p in where_parts if p])
                            st.write(f"**When:** {when} ({r['time_zone']})")
                            st.write(f"**Where:** {where}")
                            st.write(f"**Organizers:** {r['organizers'] or 'Unknown'}")
                            st.write(f"**Free:** {r['free_item'] or '[unspecified]'}")
                            st.write(f"**Details:** {r['free_details']}")
                            if r["permalink"]:
                                st.markdown(f"[Event link]({r['permalink']})")

        # ---------- PROFESSIONAL EVENTS TAB ----------
        if show_prof:
            tab_idx = tabs.index("💼 Professionally Helpful")
            with tab_objs[tab_idx]:
                st.subheader("💼 Professionally Helpful Events")
                with st.spinner("Evaluating events for professional relevance..."):
                    helpful_events, categories = process_professional_events_concurrent(
                        events,
                        model=DEFAULT_MODEL,
                        max_workers=max_workers,
                        temperature=temperature,
                    )

                st.write(f"Found **{len(helpful_events)}** professionally helpful events.")

                df_prof = pd.DataFrame(helpful_events)
                with st.expander("View all professionally helpful events as a table", expanded=True):
                    st.dataframe(df_prof, use_container_width=True)

                st.markdown("#### Grouped by Professional Category")
                for cat, rows in categories.items():
                    st.markdown(f"**{cat.upper()} ({len(rows)})**")
                    for r in rows:
                        with st.expander(r["title"]):
                            when = f"{r['date_start']} {r['time_start']}".strip()
                            st.write(f"**When:** {when}")
                            st.write(f"**Where:** {r['location']}")
                            st.write(f"**Organizers:** {r['organizers'] or 'Unknown'}")
                            st.write(f"**Why helpful:** {r['reason']}")
                            if r["link"]:
                                st.markdown(f"[Event link]({r['link']})")

                # Optional CSV download
                if helpful_events:
                    csv_prof = df_prof.to_csv(index=False).encode("utf-8")
                    st.download_button(
                        label="Download professionally helpful events as CSV",
                        data=csv_prof,
                        file_name="umich_professionally_helpful_events.csv",
                        mime="text/csv",
                    )
else:
    st.info("Configure options in the sidebar and click **Run scan** to start.")


2025-11-13 19:20:15.089 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 19:20:15.090 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 19:20:15.112 
  command:

    streamlit run /Users/mustafa/Library/Python/3.13/lib/python/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-11-13 19:20:15.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 19:20:15.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 19:20:15.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-11-13 19:20:15.114 Thread 'MainThread': missing ScriptRunContext! This warning can be 